In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Inegalitatea CHSH

*Estimare utilizare: Două minute pe un procesor Heron r2 (NOTĂ: Aceasta este doar o estimare. Timpul de rulare poate varia.)*
## Fundal
În acest tutorial, vei rula un experiment pe un calculator cuantic pentru a demonstra violarea inegalității CHSH cu primitivul Estimator.

Inegalitatea CHSH, denumită după autorii Clauser, Horne, Shimony și Holt, este folosită pentru a prova experimental teorema lui Bell (1969). Această teoremă afirmă că teoriile cu variabile ascunse locale nu pot da seama de unele consecințe ale entanglementului în mecanica cuantică. Violarea inegalității CHSH este folosită pentru a arăta că mecanica cuantică este incompatibilă cu teoriile cu variabile ascunse locale. Acesta este un experiment important pentru înțelegerea fundamentelor mecanicii cuantice.

Premiul Nobel pentru Fizică din 2022 a fost acordat lui Alain Aspect, John Clauser și Anton Zeilinger, parțial pentru munca lor de pionierat în știința informației cuantice și, în special, pentru experimentele cu fotoni entanglați care demonstrează violarea inegalităților lui Bell.
## Cerințe
Înainte de a începe acest tutorial, asigură-te că ai instalate următoarele:

* Qiskit SDK v1.0 sau ulterior, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 sau ulterior
## Configurare

In [1]:
# General
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Plotting routines
import matplotlib.pyplot as plt
import matplotlib.ticker as tck

## Pasul 1: Maparea intrărilor clasice la o problemă cuantică
Pentru acest experiment, vom crea o pereche entanglată pe care o vom măsura pe fiecare Qubit în două baze diferite. Vom eticheta bazele pentru primul Qubit cu $A$ și $a$, iar bazele pentru al doilea Qubit cu $B$ și $b$. Aceasta ne permite să calculăm cantitatea CHSH $S_1$:

$$
S_1 = A(B-b) + a(B+b).
$$

Fiecare observabil este fie $+1$, fie $-1$. Evident, unul dintre termenii $B\pm b$ trebuie să fie $0$, iar celălalt trebuie să fie $\pm 2$. Prin urmare, $S_1 = \pm 2$. Valoarea medie a lui $S_1$ trebuie să satisfacă inegalitatea:

$$
|\langle S_1 \rangle|\leq 2.
$$

Expandând $S_1$ în termenii lui $A$, $a$, $B$ și $b$ rezultă:

$$
|\langle S_1 \rangle| = |\langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle| \leq 2
$$

Poți defini o altă cantitate CHSH $S_2$:

$$
S_2 = A(B+b) - a(B-b),
$$

Aceasta conduce la o altă inegalitate:

$$
|\langle S_2 \rangle| = |\langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle| \leq 2
$$

Dacă mecanica cuantică poate fi descrisă prin teorii cu variabile ascunse locale, inegalitățile anterioare trebuie să fie adevărate. Cu toate acestea, așa cum este demonstrat în acest tutorial, aceste inegalități pot fi violate pe un calculator cuantic. Prin urmare, mecanica cuantică nu este compatibilă cu teoriile cu variabile ascunse locale.
Dacă vrei să înveți mai multă teorie, explorează [Entanglementul în acțiune](/learning/courses/basics-of-quantum-information/entanglement-in-action/chsh-game) cu John Watrous.
Vei crea o pereche entanglată între două Qubituri dintr-un calculator cuantic prin crearea stării Bell $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$. Folosind primitivul Estimator, poți obține direct valorile de așteptare necesare ($\langle AB \rangle, \langle Ab \rangle, \langle aB \rangle$ și $\langle ab \rangle$) pentru a calcula valorile de așteptare ale celor două cantități CHSH $\langle S_1\rangle$ și $\langle S_2\rangle$. Înainte de introducerea primitivului Estimator, ar fi trebuit să construiești valorile de așteptare din rezultatele măsurătorilor.

Al doilea Qubit va fi măsurat în bazele $Z$ și $X$. Primul Qubit va fi măsurat, de asemenea, în baze ortogonale, dar cu un unghi față de al doilea Qubit, pe care îl vom varia între $0$ și $2\pi$. Așa cum vei vedea, primitivul Estimator face rularea circuitelor parametrizate foarte ușoară. În loc să creezi o serie de circuite CHSH, trebuie să creezi *un singur* Circuit CHSH cu un parametru care specifică unghiul de măsurare și o serie de valori de fază pentru parametru.

În final, vei analiza rezultatele și le vei reprezenta grafic în funcție de unghiul de măsurare. Vei vedea că pentru un anumit interval de unghiuri de măsurare, valorile de așteptare ale cantităților CHSH $|\langle S_1\rangle| > 2$ sau $|\langle S_2\rangle| > 2$, ceea ce demonstrează violarea inegalității CHSH.

In [2]:
# To run on hardware, select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_kingston'

### Create a parameterized CHSH circuit

First, we write the circuit with the parameter $\theta$, which we call `theta`. The [`Estimator` primitive](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) can enormously simplify circuit building and output analysis by directly providing expectation values of observables. Many problems of interest, especially for near-term applications on noisy systems, can be formulated in terms of expectation values. `Estimator` (V2) primitive can automatically change measurement basis based on the supplied observable.

In [3]:
theta = Parameter("$\\theta$")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
chsh_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif" alt="Output of the previous code cell" />

### Crearea unui Circuit CHSH parametrizat
Mai întâi, scriem circuitul cu parametrul $\theta$, pe care îl numim `theta`. Primitivul [`Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) poate simplifica enorm construirea circuitelor și analiza ieșirilor, oferind direct valorile de așteptare ale observabilelor. Multe probleme de interes, în special pentru aplicațiile pe termen scurt pe sisteme cu zgomot, pot fi formulate în termeni de valori de așteptare. Primitivul `Estimator` (V2) poate schimba automat baza de măsurare pe baza observabilului furnizat.

In [4]:
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
# Phases need to be expressed as list of lists in order to work
individual_phases = [[ph] for ph in phases]

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif)

### Crearea unei liste de valori de fază ce vor fi atribuite ulterior
După crearea circuitului CHSH parametrizat, vei crea o listă de valori de fază care vor fi atribuite circuitului în pasul următor. Poți folosi codul următor pentru a crea o listă de 21 de valori de fază cuprinse între $0$ și $2 \pi$ cu spațiere egală, adică $0$, $0.1 \pi$, $0.2 \pi$, ..., $1.9 \pi$, $2 \pi$.

In [5]:
# <CHSH1> = <AB> - <Ab> + <aB> + <ab> -> <ZZ> - <ZX> + <XZ> + <XX>
observable1 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)]
)

# <CHSH2> = <AB> + <Ab> - <aB> + <ab> -> <ZZ> + <ZX> - <XZ> + <XX>
observable2 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)]
)

### Observabile
Acum avem nevoie de observabile din care să calculăm valorile de așteptare. În cazul nostru, ne uităm la baze ortogonale pentru fiecare Qubit, lăsând rotația $Y-$ parametrizată pentru primul Qubit să varieze baza de măsurare aproape continuu față de baza celui de-al doilea Qubit. Vom alege, prin urmare, observabilele $ZZ$, $ZX$, $XZ$ și $XX$.

In [6]:
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)

chsh_isa_circuit = pm.run(chsh_circuit)
chsh_isa_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif" alt="Output of the previous code cell" />

## Pasul 2: Optimizarea problemei pentru execuția pe hardware cuantic

Pentru a reduce timpul total de execuție a joburilor, primitivele V2 acceptă doar circuite și observabile care respectă instrucțiunile și conectivitatea suportate de sistemul țintă (denumite circuite și observabile cu arhitectura setului de instrucțiuni (ISA)).

### Circuit ISA

In [7]:
isa_observable1 = observable1.apply_layout(layout=chsh_isa_circuit.layout)
isa_observable2 = observable2.apply_layout(layout=chsh_isa_circuit.layout)

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif)

### Observabile ISA

Similar, trebuie să transformăm observabilele pentru a le face compatibile cu Backend-ul înainte de a rula joburi cu [`Runtime Estimator V2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run). Putem efectua transformarea folosind metoda `apply_layout` a obiectului `SparsePauliOp`.

In [8]:
# To run on a local simulator:
# Use the StatevectorEstimator from qiskit.primitives instead.

estimator = Estimator(mode=backend)

pub = (
    chsh_isa_circuit,  # ISA circuit
    [[isa_observable1], [isa_observable2]],  # ISA Observables
    individual_phases,  # Parameter values
)

job_result = estimator.run(pubs=[pub]).result()

## Pasul 3: Executarea cu primitivele Qiskit
Pentru a executa întregul experiment într-un singur apel la [`Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2).
Putem crea un primitiv Qiskit Runtime [`Estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) pentru a calcula valorile noastre de așteptare. Metoda `EstimatorV2.run()` primește un iterabil de `blocuri unificate primitive (PUBs)`. Fiecare PUB este un iterabil în formatul `(circuit, observables, parameter_values: Optional, precision: Optional)`.

In [9]:
chsh1_est = job_result[0].data.evs[0]
chsh2_est = job_result[0].data.evs[1]

In [10]:
fig, ax = plt.subplots(figsize=(10, 6))

# results from hardware
ax.plot(phases / np.pi, chsh1_est, "o-", label="CHSH1", zorder=3)
ax.plot(phases / np.pi, chsh2_est, "o-", label="CHSH2", zorder=3)

# classical bound +-2
ax.axhline(y=2, color="0.9", linestyle="--")
ax.axhline(y=-2, color="0.9", linestyle="--")

# quantum bound, +-2√2
ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
ax.fill_between(phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7)

# set x tick labels to the unit of pi
ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

# set labels, and legend
plt.xlabel("Theta")
plt.ylabel("CHSH witness")
plt.legend()
plt.show()

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/f6267448-0.avif" alt="Output of the previous code cell" />

In the figure, the lines and gray areas delimit the bounds; the outer-most (dash-dotted) lines delimit the quantum-bounds ($\pm 2$), whereas the inner (dashed) lines delimit the classical bounds ($\pm 2\sqrt{2}$). You can see that there are regions where the CHSH witness quantities exceeds the classical bounds. Congratulations! You have successfully demonstrated the violation of CHSH inequality in a real quantum system!

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_3xxAgm1SF1wGp9k)